# Small CNN
Small CNN trained on 128x128 aspect-ratio-preserving padded RGB inputs.
Loads the frozen split from `data/splits/patient_split.csv`

**Note:** `src/preprocessing.py` (Shashank's) is still a stub, so this notebook has a local, temporary
`load_and_pad` placeholder below. Once `src/preprocessing.py` is implemented, delete the placeholder
cell and switch the import back to `from src.preprocessing import load_and_pad`.

In [7]:
import sys
sys.path.append("..")

import kagglehub
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Temporary local preprocessing (remove once `src/preprocessing.py` is implemented)

In [8]:
IMG_SIZE = 128

def load_and_pad(path: str, size: int = IMG_SIZE) -> np.ndarray:
    """TEMPORARY placeholder — mirrors what src/preprocessing.load_and_pad should do.
    Loads an RGB image, preserves aspect ratio, pads to a (size, size) square.
    """
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    img = img.resize((new_w, new_h), Image.BILINEAR)

    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    paste_x = (size - new_w) // 2
    paste_y = (size - new_h) // 2
    canvas.paste(img, (paste_x, paste_y))

    arr = np.asarray(canvas, dtype=np.float32) / 255.0  # scale to [0, 1]
    return arr

## Load the frozen split and build a Dataset/DataLoader
Images are loaded lazily per-batch.

In [ ]:
manifest = pd.read_csv("../data/splits/patient_split.csv")
print(manifest["split"].value_counts())

# The manifest only stores filename + category, not an absolute path (those differ per machine).
# The path is rebuild the same way as 00_eda.ipynb (<kagglehub download root>/<category>/<filename>) 
download_path = Path(kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria"))
data_dir = download_path / "cell_images"
if (data_dir / "cell_images").exists():
    data_dir = data_dir / "cell_images"

CLASS_TO_IDX = {"Uninfected": 0, "Parasitized": 1}


class CellDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        path = self.data_dir / row["category"] / row["filename"]

        # Load the padded image as a (height, width, channels) array, then rearrange it to
        # (channels, height, width), since PyTorch's conv layers expect the channel dimension first.
        img = load_and_pad(str(path))
        img = torch.from_numpy(img).permute(2, 0, 1)

        label = CLASS_TO_IDX[row["category"]]
        return img, label


train_ds = CellDataset(manifest[manifest["split"] == "train"], data_dir)
val_ds = CellDataset(manifest[manifest["split"] == "val"], data_dir)
test_ds = CellDataset(manifest[manifest["split"] == "test"], data_dir)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Small CNN

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        
        # Conv2d(in_channels, out_channels, kernel_size, padding): in_channels must match the
        # previous layer's out_channels (3 for the RGB input); padding=1 with a 3x3 kernel keeps
        # the spatial size unchanged, so only MaxPool2d(2) halves it (128 -> 64 -> 32 -> 16).
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 128 -> 64
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64 -> 32
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 32 -> 16
        )

        # Flatten turns the (64, 16, 16) feature map into one vector of 64*16*16 values, which
        # Linear(in_features, out_features) maps down to 128, then to num_classes.
        # Dropout(0.3) randomly zeroes 30% of activations during training to reduce overfitting.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Training loop (validation only - freeze the protocol before touching test_loader)

In [11]:
def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, correct, n = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        n += images.size(0)

    return total_loss / n, correct / n


NUM_EPOCHS = 5
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

Epoch 1: train_loss=0.6275 train_acc=0.6345 val_loss=0.3750 val_acc=0.8490
Epoch 2: train_loss=0.2082 train_acc=0.9291 val_loss=0.1756 val_acc=0.9494
Epoch 3: train_loss=0.1556 train_acc=0.9512 val_loss=0.1697 val_acc=0.9489
Epoch 4: train_loss=0.1404 train_acc=0.9560 val_loss=0.1644 val_acc=0.9482
Epoch 5: train_loss=0.1331 train_acc=0.9571 val_loss=0.1636 val_acc=0.9489
